# Training

In [2]:
import torch
import pandas as pd
import numpy as np
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, TrainingArguments, Trainer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

## Data Preparation

In [3]:
# Load merged data from previous notebook
merged_data = pd.read_csv("../datasets/processed/merged_data.csv") 

# Clean text data
merged_data['full_text'] = merged_data['designation'] + " " + merged_data['description'].fillna('')

# Convert labels to categorical codes
labels = merged_data['prdtypecode'].astype('category').cat.codes
num_labels = len(merged_data['prdtypecode'].unique())

## Train & Validation

In [4]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    merged_data['full_text'].tolist(),
    labels.tolist(),
    test_size=0.2,
    random_state=42,
    stratify=labels
)

## Tokenization

In [5]:
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=128
)

val_encodings = tokenizer(
    val_texts,
    truncation=True,
    padding=True,
    max_length=128
)

## Dataset Class

In [6]:
class ProductDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = ProductDataset(train_encodings, train_labels)
val_dataset = ProductDataset(val_encodings, val_labels)

## Model Configuration

In [16]:
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=num_labels
)

training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ImportError: Using the `Trainer` with `PyTorch` requires `accelerate>=0.26.0`: Please run `pip install transformers[torch]` or `pip install 'accelerate>={ACCELERATE_MIN_VERSION}'`

## Training & Evaluation

In [ ]:
def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {
        'accuracy': accuracy_score(p.label_ids, preds),
        'report': classification_report(p.label_ids, preds, output_dict=True)
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

## Save Model

In [ ]:
model.save_pretrained("../models/text_classifier")
tokenizer.save_pretrained("../models/text_classifier")